# CLIP Layer-wise Embeddings — SNLI-VE

Tests the layer-wise extraction pipeline (`scripts/extract_snli_ve_clip_layers.py`).

Sections:
0. Setup
1. Load dataset + subsample
2. Load CLIP model
3. Extract layer-wise embeddings
4. Verify `.npz` output
5. PCA per layer — vision tower
6. PCA per layer — text tower
7. Layer-wise linear separability

## 0 · Setup

In [ ]:
import random
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(REPO_ROOT))

from datasets import load_dataset
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

from features_extraction import save_features
from features_extraction.device import DeviceManager

# ── constants ──────────────────────────────────────────────────────────────
SEED        = 42
N_PER_CLASS = 1000          # smoke-test size; change to 1000 for full run
HF_ID       = "pingzhili/snli-ve"
SPLIT       = "train"
LABEL_COL   = "gold_label"
TEXT_COL    = "sentence"
MODEL_NAME  = "openai/clip-vit-base-patch32"
BATCH_SIZE  = 32
OUTPUT_PATH = REPO_ROOT / "data" / "embeddings" / "snli_ve_clip_layers_smoke.npz"

LABEL_MAP = {"entailment": 0, "neutral": 1, "contradiction": 2}
LABEL_NAMES = ["entailment", "neutral", "contradiction"]
COLORS = ["#e41a1c", "#377eb8", "#4daf4a"]

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Output:    {OUTPUT_PATH}")

## 1 · Load dataset + subsample

In [ ]:
INVALID_LABELS = {"-", ""}

def resolve_label(raw):
    if isinstance(raw, int):
        return raw if raw >= 0 else None
    s = str(raw).strip().lower()
    if s in INVALID_LABELS or s == "-1":
        return None
    return LABEL_MAP.get(s)

def balanced_subsample(dataset, n_per_class, seed, label_col):
    rng = random.Random(seed)
    label_to_indices = defaultdict(list)
    for idx, raw in enumerate(dataset[label_col]):
        lbl = resolve_label(raw)
        if lbl is not None:
            label_to_indices[lbl].append(idx)
    selected = []
    for lbl in sorted(label_to_indices):
        chosen = rng.sample(label_to_indices[lbl], n_per_class)
        selected.extend(chosen)
    selected.sort()
    return dataset.select(selected)

print(f"Loading {HF_ID} ({SPLIT})...")
raw = load_dataset(HF_ID, split=SPLIT)
print(f"  {len(raw)} rows | columns: {raw.column_names}")

subset = balanced_subsample(raw, N_PER_CLASS, SEED, LABEL_COL)
labels_raw = [resolve_label(v) for v in subset[LABEL_COL]]
unique, counts = np.unique(labels_raw, return_counts=True)
print(f"  subset: {len(subset)} samples | {dict(zip(unique.tolist(), counts.tolist()))}")

## 2 · Load CLIP model

In [ ]:
print(f"Loading {MODEL_NAME}...")
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME)

device = DeviceManager.resolve("auto")
DeviceManager.prepare_model(model, device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  device: {device}")
print(f"  total params:     {total_params:,}")
print(f"  trainable params: {trainable:,}  (should be 0 — frozen)")

## 3 · Extract layer-wise embeddings

In [ ]:
GROUP_COL = "Flickr30K_ID"   # image identifier — used for grouped CV later

def coerce_to_pil(img_field):
    if not isinstance(img_field, Image.Image):
        raise TypeError(f"Unexpected image type: {type(img_field).__name__}")
    return img_field.convert("RGB") if img_field.mode != "RGB" else img_field

def l2_normalize(x):
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-12)

def extract_layer_embeddings(model, processor, dataset, device, batch_size,
                             label_col, text_col, group_col):
    """Extracts CLS-pooled vision and EOS-pooled text states per layer.

    Vision pooling: position 0 (CLS).
    Text   pooling: per-sample EOS token (input_ids.argmax) — CLIP's actual
                    pooling position; position 0 (SOS) is identical across samples.

    Also returns a groups array (image IDs) aligned with the labels and embeddings
    so downstream CV can keep all rows of the same image in the same fold.
    """
    vision_chunks = defaultdict(list)
    text_chunks   = defaultdict(list)
    labels_list   = []
    groups_list   = []
    n = len(dataset)
    n_batches = (n + batch_size - 1) // batch_size

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        rows = dataset[start:end]

        images, texts, labels, groups = [], [], [], []
        for img_f, hyp, raw_lab, grp in zip(
            rows["image"], rows[text_col], rows[label_col], rows[group_col]
        ):
            try:
                images.append(coerce_to_pil(img_f))
            except Exception:
                continue
            lbl = resolve_label(raw_lab)
            if lbl is None:
                images.pop()
                continue
            texts.append(hyp)
            labels.append(lbl)
            groups.append(grp)

        if not images:
            continue

        inputs = processor(text=texts, images=images, return_tensors="pt",
                           padding=True, truncation=True)
        pixel_values   = inputs["pixel_values"].to(device)
        input_ids      = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        # Vision tower → CLS token (position 0)
        vision_out = model.vision_model(pixel_values=pixel_values,
                                        output_hidden_states=True)
        for layer_index, hidden_state in enumerate(vision_out.hidden_states):
            cls_vector = l2_normalize(hidden_state[:, 0, :])
            vision_chunks[f"layer_{layer_index:02d}"].append(
                cls_vector.detach().cpu().to(torch.float32)
            )

        # Text tower → EOS token per sample
        text_out = model.text_model(input_ids=input_ids,
                                    attention_mask=attention_mask,
                                    output_hidden_states=True)
        batch_size_actual = input_ids.shape[0]
        eos_positions = input_ids.to(torch.int).argmax(dim=-1)
        batch_indices = torch.arange(batch_size_actual, device=input_ids.device)
        for layer_index, hidden_state in enumerate(text_out.hidden_states):
            eos_vector = l2_normalize(hidden_state[batch_indices, eos_positions])
            text_chunks[f"layer_{layer_index:02d}"].append(
                eos_vector.detach().cpu().to(torch.float32)
            )

        labels_list.extend(labels)
        groups_list.extend(groups)

        batch_idx = start // batch_size + 1
        if batch_idx % 5 == 0 or batch_idx == n_batches:
            so_far = sum(c.shape[0] for c in vision_chunks["layer_00"])
            print(f"  batch {batch_idx}/{n_batches} — accumulated: {so_far}")

    vision_layers = {k: torch.cat(v, 0).numpy() for k, v in vision_chunks.items()}
    text_layers   = {k: torch.cat(v, 0).numpy() for k, v in text_chunks.items()}
    labels_arr    = np.asarray(labels_list, dtype=np.int64)
    groups_arr    = np.asarray(groups_list)
    return vision_layers, text_layers, labels_arr, groups_arr

print(f"Extracting layer-wise embeddings ({N_PER_CLASS * 3} samples)...")
with torch.no_grad():
    vision_layers, text_layers, labels_arr, groups_arr = extract_layer_embeddings(
        model, processor, subset, device, BATCH_SIZE, LABEL_COL, TEXT_COL, GROUP_COL
    )

# Sanity check: text embeddings must NOT all be identical
v_std = vision_layers["layer_06"].std(axis=0).mean()
t_std = text_layers["layer_06"].std(axis=0).mean()
print(f"\nMean per-dimension std at layer 06 — vision: {v_std:.4f}, text: {t_std:.4f}")
print("(both should be > 0; near-zero text std means pooling is collapsing samples)")

n_unique_images = len(np.unique(groups_arr))
print(f"\nGroups: {len(groups_arr)} rows from {n_unique_images} unique images")
print(f"  → {len(groups_arr) / n_unique_images:.2f} hypotheses per image (avg)")

print(f"\nVision layers: {len(vision_layers)}")
for k, v in sorted(vision_layers.items()):
    print(f"  vision_{k}: {v.shape}")
print(f"\nText layers: {len(text_layers)}")
for k, v in sorted(text_layers.items()):
    print(f"  text_{k}: {v.shape}")
print(f"\nLabels: {labels_arr.shape}  unique={np.unique(labels_arr, return_counts=True)}")


## 4 · Verify `.npz` output

In [ ]:
representations = {}
for k, v in vision_layers.items():
    representations[f"vision_{k}"] = v
for k, v in text_layers.items():
    representations[f"text_{k}"] = v

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
save_features(representations, labels_arr, str(OUTPUT_PATH))
print(f"Saved: {OUTPUT_PATH}  ({OUTPUT_PATH.stat().st_size / 1e6:.1f} MB)")

# ── reload and verify ──────────────────────────────────────────────────────
f = np.load(OUTPUT_PATH)
print(f"\nKeys in .npz: {len(f.files)}")

n_vision_keys = len([k for k in f.files if k.startswith("features_vision_layer_")])
n_text_keys   = len([k for k in f.files if k.startswith("features_text_layer_")])
print(f"  vision layer keys: {n_vision_keys}")
print(f"  text layer keys:   {n_text_keys}")

N = N_PER_CLASS * 3
assert f["features_vision_layer_00"].shape == (N, 768), "vision shape mismatch"
assert f["features_text_layer_00"].shape   == (N, 512), "text shape mismatch"
assert f["labels"].shape == (N,), "labels shape mismatch"
_, cnts = np.unique(f["labels"], return_counts=True)
assert list(cnts) == [N_PER_CLASS] * 3, "class balance mismatch"

# L2 norms should be ≈ 1.0
vision_norms = np.linalg.norm(f["features_vision_layer_06"], axis=1)
text_norms   = np.linalg.norm(f["features_text_layer_06"], axis=1)
print(f"\nL2 norm vision_layer_06: mean={vision_norms.mean():.6f}, std={vision_norms.std():.6f}")
print(f"L2 norm text_layer_06:   mean={text_norms.mean():.6f}, std={text_norms.std():.6f}")

print("\nAll assertions passed.")

## 5 · PCA per layer — vision tower

In [ ]:
n_layers = len(vision_layers)
n_cols = 5
n_rows = (n_layers + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, key in enumerate(sorted(vision_layers.keys())):
    layer_num = int(key.split("_")[1])
    X = vision_layers[key]
    pca = PCA(n_components=2, random_state=SEED)
    X2 = pca.fit_transform(X)
    ax = axes[i]
    for lbl, name, color in zip([0, 1, 2], LABEL_NAMES, COLORS):
        mask = labels_arr == lbl
        ax.scatter(X2[mask, 0], X2[mask, 1], c=color, label=name, alpha=0.7, s=20)
    ax.set_title(f"Vision layer {layer_num:02d}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    var = pca.explained_variance_ratio_.sum() * 100
    ax.set_xlabel(f"var={var:.1f}%", fontsize=8)

# legend on last used axis
axes[n_layers - 1].legend(fontsize=8, loc="best")

# hide unused subplots
for j in range(n_layers, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("PCA (2D) — CLIP Vision Tower, Layer-wise CLS Embeddings", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPO_ROOT / "data" / "embeddings" / "snli_ve_clip_vision_layers_pca.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Vision PCA saved.")

## 6 · PCA per layer — text tower

In [ ]:
n_text = len(text_layers)
n_rows_t = (n_text + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows_t, n_cols, figsize=(4 * n_cols, 4 * n_rows_t))
axes = axes.flatten()

for i, key in enumerate(sorted(text_layers.keys())):
    layer_num = int(key.split("_")[1])
    X = text_layers[key]
    pca = PCA(n_components=2, random_state=SEED)
    X2 = pca.fit_transform(X)
    ax = axes[i]
    for lbl, name, color in zip([0, 1, 2], LABEL_NAMES, COLORS):
        mask = labels_arr == lbl
        ax.scatter(X2[mask, 0], X2[mask, 1], c=color, label=name, alpha=0.7, s=20)
    ax.set_title(f"Text layer {layer_num:02d}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    var = pca.explained_variance_ratio_.sum() * 100
    ax.set_xlabel(f"var={var:.1f}%", fontsize=8)

axes[n_text - 1].legend(fontsize=8, loc="best")

for j in range(n_text, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("PCA (2D) — CLIP Text Tower, Layer-wise CLS Embeddings", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPO_ROOT / "data" / "embeddings" / "snli_ve_clip_text_layers_pca.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Text PCA saved.")

## 7 · Layer-wise linear separability (grouped by image)

The same image appears in multiple rows of SNLI-VE paired with different
hypotheses (and different labels). With plain k-fold CV, the model can memorize
per-image label patterns instead of learning the actual image–text matching
signal. We use `GroupKFold` keyed on `Flickr30K_ID` so all rows of the same
image stay in the same fold.

Three probes per layer with logistic regression on the raw L2-normalized
embeddings (no PCA):

1. **Vision tower** — CLS-pooled image embedding only.
2. **Text tower** — EOS-pooled hypothesis embedding only.
3. **Joint pairwise** — `concat([V_i, T_i, |V_i - T_i|, V_i * T_i])`. Entailment
   is a multimodal relation, so this curve should carry the actual signal.

Vision is 768-d while text is 512-d. Element-wise ops `|V - T|` and `V * T`
require matching widths, so vision is truncated to the first 512 dims for the
joint construction. A dashed line marks chance (1/3).


In [ ]:
from sklearn.model_selection import GroupKFold

# Same image appears in multiple rows with different hypotheses/labels.
# GroupKFold keeps all rows of the same image in the same fold, so the probe
# can't memorize per-image label patterns and is forced to learn the actual
# image–text matching signal.
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)


def layer_accuracy(layers_dict, labels, groups, cv, seed=42):
    """Returns list of (layer_idx, mean_grouped_cv_accuracy) per layer."""
    results = []
    for key in sorted(layers_dict.keys()):
        layer_num = int(key.split("_")[1])
        X = layers_dict[key]
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(max_iter=1000, C=1.0, random_state=seed)),
        ])
        scores = cross_val_score(pipe, X, labels, groups=groups, cv=cv, scoring="accuracy")
        results.append((layer_num, scores.mean()))
    return results


def joint_accuracy(vision_layers, text_layers, labels, groups, cv, seed=42):
    """Probes concat([V_i, T_i, |V_i - T_i|, V_i * T_i]) per matching layer pair."""
    results = []
    common_keys = sorted(set(vision_layers) & set(text_layers))
    for key in common_keys:
        layer_num = int(key.split("_")[1])
        v = vision_layers[key]
        t = text_layers[key]
        # Vision is 768-d, text is 512-d → element-wise ops require matching dims.
        # Truncate vision to text's width so |v - t| and v * t are well-defined.
        d = min(v.shape[1], t.shape[1])
        v_trunc = v[:, :d]
        t_trunc = t[:, :d]
        joint = np.concatenate(
            [v_trunc, t_trunc, np.abs(v_trunc - t_trunc), v_trunc * t_trunc],
            axis=1,
        )
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(max_iter=1000, C=1.0, random_state=seed)),
        ])
        scores = cross_val_score(pipe, joint, labels, groups=groups, cv=cv, scoring="accuracy")
        results.append((layer_num, scores.mean()))
    return results


print(f"GroupKFold setup: {N_SPLITS} folds | "
      f"{len(np.unique(groups_arr))} unique images across {len(labels_arr)} rows")

print("Computing layer-wise linear separability (vision)...")
vision_acc = layer_accuracy(vision_layers, labels_arr, groups_arr, gkf)
print("Computing layer-wise linear separability (text)...")
text_acc   = layer_accuracy(text_layers, labels_arr, groups_arr, gkf)
print("Computing joint pairwise separability per layer...")
joint_acc  = joint_accuracy(vision_layers, text_layers, labels_arr, groups_arr, gkf)

vision_indices, vision_scores = zip(*vision_acc)
text_indices,   text_scores   = zip(*text_acc)
joint_indices,  joint_scores  = zip(*joint_acc)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(vision_indices, vision_scores, marker="o", label="Vision tower",        color="steelblue")
ax.plot(text_indices,   text_scores,   marker="s", label="Text tower",          color="darkorange")
ax.plot(joint_indices,  joint_scores,  marker="^", label="Joint pairwise (V,T,|V-T|,V*T)", color="seagreen")
ax.axhline(1 / 3, color="gray", linestyle="--", linewidth=1, label="Chance (1/3)")
ax.set_xlabel("Layer index (0 = embedding, 1–12 = transformer blocks)")
ax.set_ylabel(f"{N_SPLITS}-fold GroupKFold CV accuracy")
ax.set_title("Linear separability per CLIP layer — SNLI-VE (LogReg, grouped by image)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(list(vision_indices))
plt.tight_layout()
plt.savefig(REPO_ROOT / "data" / "embeddings" / "snli_ve_clip_layers_separability.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("\nVision tower — accuracy per layer:")
for idx, acc in vision_acc:
    print(f"  layer {idx:02d}: {acc:.4f}")
print("\nText tower — accuracy per layer:")
for idx, acc in text_acc:
    print(f"  layer {idx:02d}: {acc:.4f}")
print("\nJoint pairwise — accuracy per layer:")
for idx, acc in joint_acc:
    print(f"  layer {idx:02d}: {acc:.4f}")
